# Experimento de Intensidad Emocional - Qwen:14b
Este notebook instala Ollama, descarga el modelo `qwen:14b` y ejecuta el experimento de intensidad emocional. 

**Importante:** Recuerda cambiar el entorno de ejecución a una GPU (como la T4 u otra disponible en Google Colab) en la pestaña `Entorno de ejecución > Cambiar tipo de entorno de ejecución` para que el modelo responda más rápido.

In [ ]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama pandas

In [ ]:
import subprocess
import time

print("Iniciando servidor Ollama en segundo plano...")
process = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)

print("Descargando modelo qwen:14b (Esto puede tardar unos minutos)...")
!ollama pull qwen:14b
print("¡Modelo listo!")

In [ ]:
import ollama
import pandas as pd
import time
import os

# CONFIG
MODELO = "qwen:14b"

INTENSIDAD_EMOCIONAL = {
    "neutral": "Explícame cómo funciona una red neuronal.",
    "frustracion_baja": "Me siento un poco confundido, explícame cómo funciona una red neuronal.",
    "frustracion_media": "Estoy frustrado, no entiendo nada, explícame cómo funciona una red neuronal.",
    "frustracion_alta": "¡Estoy harto y sumamente frustrado! ¡Es imposible entender esto! Explícame cómo funciona una red neuronal.",
    "urgencia_baja": "Cuando tengas un momento, explícame cómo funciona una red neuronal.",
    "urgencia_media": "Necesito que me expliques cómo funciona una red neuronal pronto.",
    "urgencia_alta": "¡Rápido, es de extrema urgencia! ¡Necesito entender cómo funciona una red neuronal AHORA MISMO!",
    "tristeza_baja": "Me siento un poco desanimado hoy. Explícame cómo funciona una red neuronal.",
    "tristeza_media": "Estoy triste y me cuesta mucho concentrarme. Explícame cómo funciona una red neuronal.",
    "tristeza_alta": "Estoy totalmente devastado y sin fuerzas para nada. Solo explícame cómo funciona una red neuronal.",
    "entusiasmo_bajo": "Me parece interesante, explícame cómo funciona una red neuronal.",
    "entusiasmo_medio": "¡Qué genial! Me entusiasma aprender, explícame cómo funciona una red neuronal.",
    "entusiasmo_alto": "¡Estoy increíblemente emocionado y feliz por aprender esto! ¡Explícame cómo funciona una red neuronal!"
}

REPETICIONES = 5

PARAMS = {
    "temperature": 0.7,
    "num_predict": 200
}

# FUNCIÓN MODELO
def query_model(prompt):
    try:
        response = ollama.chat(
            model=MODELO,
            messages=[{"role": "user", "content": prompt}],
            options=PARAMS
        )
        return response["message"]["content"]
    except Exception as e:
        print(f"Error al consultar el modelo: {e}")
        return "Error"

# MÉTRICAS
def evaluar(texto):
    if texto == "Error":
        return {"longitud": 0, "diversidad": 0.0}
        
    palabras = texto.split()
    return {
        "longitud": len(palabras),
        "diversidad": len(set(palabras)) / len(palabras) if palabras else 0
    }

# EXPERIMENTO
resultados = []
print(f"Iniciando experimento de Intensidad Emocional con {MODELO}...\n")

for nivel, prompt in INTENSIDAD_EMOCIONAL.items():
    for i in range(REPETICIONES):
        print(f"{nivel} | iter {i+1}")
        respuesta = query_model(prompt)
        print(f"Respuesta:\n{respuesta}\n{'-'*40}")
        metricas = evaluar(respuesta)
        resultados.append({
            "intensidad": nivel,
            "iteracion": i,
            "prompt": prompt,
            "respuesta": respuesta,
            **metricas
        })
        time.sleep(1)

# GUARDAR
out_dir = "dataset_experimento"
os.makedirs(out_dir, exist_ok=True)
archivo_salida = os.path.join(out_dir, f"{MODELO.replace(':', '_')}_intensidad_experimento.csv")

df = pd.DataFrame(resultados)
df.to_csv(archivo_salida, index=False)
print(f"\n¡Experimento finalizado! Datos guardados en: {archivo_salida}")
df.head()

In [ ]:
from google.colab import files

# Descargar el archivo CSV a tu máquina local una vez terminado
files.download(archivo_salida)